In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# =========================
# MODEL
# =========================
model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Gemma requires pad token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# =========================
# LoRA
# =========================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# =========================
# DATA
# =========================
data = [
    {
        "text": "User: What is AI?\nAssistant: AI is artificial intelligence."
    },
    {
        "text": "User: What is FFT?\nAssistant: FFT is fast Fourier transform."
    },
]

dataset = Dataset.from_list(data)

# =========================
# TOKENIZE
# =========================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    # IMPORTANT: labels required for causal LM loss
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize)

# Remove original text column
dataset = dataset.remove_columns(["text"])

# =========================
# DATA COLLATOR
# =========================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# =========================
# TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./gemma-lora",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# =========================
# TRAIN
# =========================
trainer.train()

# =========================
# SAVE
# =========================
model.save_pretrained("./gemma-lora")
tokenizer.save_pretrained("./gemma-lora")

print("Training complete!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

trainable params: 921,600 || all params: 2,507,094,016 || trainable%: 0.0368


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,10.019300
2,10.418400


Training complete!


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = "google/gemma-2b-it"
adapter_path = "./gemma-lora"
output_path = "./gemma-merged"

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-2b-it",
    use_fast=False
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

model.save_pretrained(output_path, safe_serialization=True)
tokenizer.save_pretrained(output_path)

print("Saved merged model to ./gemma-merged")

/Users/br4c3/Projects/gemma4hackerton/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/br4c3/Projects/gemma4hackerton/.venv/lib/python3.9/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
Saved merged model to ./gemma-merged


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "google/gemma-2b-it"
adapter_path = "./gemma-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

prompt = "User: What is FFT?\nAssistant:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

User: What is FFT?
Assistant: Sure. Here's a breakdown of what the Fast Fourier Transform (FFT) is:

**What is it?**

- An algorithm that efficiently computes the Discrete Fourier Transform (DFT) of a discrete signal.
- A mathematical tool that expresses a signal as a sum of its constituent frequencies.
- A widely used technique in various fields, including signal processing, image processing, and communications
